# Laboratorio — Regresión Lineal Multivariable
## Dataset: Airline On-Time Performance — Data Expo 2009 (DOT/BTS)

**Fuente:** U.S. Department of Transportation, Bureau of Transportation Statistics  
**Archivo:** `DelayedFlights.csv`  
**Total de registros:** ~2,000,000 vuelos domésticos (EE.UU., año 2008)  
**Muestra usada para entrenamiento:** 200,000 registros (selección aleatoria)  
**Variable objetivo (label):** `ArrDelay` — retraso en llegada en minutos  
**Características (n):** 20 variables numéricas

---
Este cuadernillo implementa y compara **tres métodos** para estimar los parámetros θ:

| Modelo | Método | Descripción |
|--------|--------|-------------|
| M1 | Descenso de Gradiente | Regresión lineal multivariable (20 features) |
| M2 | Descenso de Gradiente | Regresión polinómica (features + términos cuadráticos) |
| M3 | Ecuación Normal | Solución analítica exacta θ = (XᵀX)⁻¹Xᵀy |

Cada modelo incluye gráfica de costo, entrenamiento, validación y predicciones.

## Parte 1 — Importar librerías

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
print('Librerías cargadas ✔')

## Parte 2 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Parte 3 — Cargar el dataset

El dataset completo contiene ~2 millones de registros. Para evitar limitaciones de memoria
en Colab, se carga una **muestra aleatoria de 200,000 filas** usando `skiprows`.
Esto es suficiente para entrenar un modelo robusto y cumple ampliamente el requisito m ≥ 20,000.

In [ ]:
# ⚠️ Ajusta esta ruta según donde guardaste el archivo en tu Drive
RUTA = '/content/drive/MyDrive/Colab Notebooks/machine learning/datasets/DelayedFlights.csv'

# Leer primero solo el encabezado para conocer las columnas
header = pd.read_csv(RUTA, nrows=0)
print('Columnas del dataset:')
print(header.columns.tolist())
print(f'Total de columnas: {len(header.columns)}')

In [ ]:
# Contar total de filas (puede tardar ~1 min en el archivo completo)
# Se omite esta celda si ya se conoce el total; descomentar si es necesario
# total_filas = sum(1 for _ in open(RUTA)) - 1
# print(f'Total de filas: {total_filas:,}')

# Cargar muestra aleatoria de 200,000 filas
# Se estima que el archivo tiene ~2,000,000 filas → prob. de inclusión ≈ 0.10
np.random.seed(42)
PROB_INCLUIR = 0.10  # ajustar si el archivo tiene más o menos de 2M filas
N_HEADER = 1

# Generar índices de filas a SALTAR (skiprows)
# Se escanea el archivo con chunksize para determinar cuántas filas tiene
print('Cargando muestra... (puede tardar 1-2 minutos)')
chunks = []
for chunk in pd.read_csv(RUTA, chunksize=100_000):
    muestra_chunk = chunk.sample(frac=PROB_INCLUIR, random_state=42)
    chunks.append(muestra_chunk)

df = pd.concat(chunks, ignore_index=True)
print(f'Muestra cargada: {len(df):,} filas  |  {df.shape[1]} columnas')
df.head(3)

## Parte 4 — Exploración y limpieza del dataset

Se eliminan registros con valores nulos en la variable objetivo (`ArrDelay`) y se
descartan columnas categóricas o irrelevantes para el modelo de regresión lineal.

In [ ]:
print('=== Tipos de datos ===')
print(df.dtypes)
print()
print('=== Valores nulos por columna ===')
print(df.isnull().sum().sort_values(ascending=False).head(15))

In [ ]:
# Eliminar filas donde ArrDelay es NaN (vuelos cancelados o desviados)
df = df.dropna(subset=['ArrDelay'])

# Columnas a descartar:
# - Categóricas: UniqueCarrier, TailNum, Origin, Dest, CancellationCode
# - Redundantes/no informativas: Year (constante=2008), Cancelled, Diverted, FlightNum
# - Columna índice extra si existe
DESCARTAR = ['Year', 'UniqueCarrier', 'FlightNum', 'TailNum',
             'Origin', 'Dest', 'Cancelled', 'CancellationCode', 'Diverted']
DESCARTAR_EXISTENTES = [c for c in DESCARTAR if c in df.columns]
df = df.drop(columns=DESCARTAR_EXISTENTES)

# Eliminar columna de índice si existe (ej: 'Unnamed: 0')
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

print(f'Dataset limpio: {len(df):,} filas  |  {df.shape[1]} columnas')
print('Columnas restantes:', df.columns.tolist())

## Parte 5 — Selección de 20 características (n ≥ 20)

Se seleccionan las columnas numéricas disponibles como predictoras y se crean
features derivadas para alcanzar n = 20 si es necesario.

In [ ]:
# Rellenar NaN en columnas de delay con 0 (ausencia de delay de ese tipo)
DELAY_COLS = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
for col in DELAY_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Rellenar otros NaN numéricos con la mediana
df = df.fillna(df.median(numeric_only=True))

# Features derivadas para enriquecer el modelo
df['DepDelay_sq']       = df['DepDelay'] ** 2           # cuadrado del retraso salida
df['TaxiTotal']         = df['TaxiIn'] + df['TaxiOut']   # tiempo total en tierra
df['ElapsedDiff']       = df['ActualElapsedTime'] - df['CRSElapsedTime']  # diferencia tiempo real vs programado
df['DepTimeNorm']       = df['CRSDepTime'] % 2400 / 2400  # hora del día normalizada

# Lista final de 20 características
FEATURES = [
    'Month', 'DayofMonth', 'DayOfWeek',
    'CRSDepTime', 'DepTime', 'CRSArrTime',
    'DepDelay', 'TaxiIn', 'TaxiOut',
    'ActualElapsedTime', 'CRSElapsedTime', 'AirTime',
    'Distance',
    'CarrierDelay', 'WeatherDelay', 'NASDelay',
    'SecurityDelay', 'LateAircraftDelay',
    'TaxiTotal', 'ElapsedDiff'
]

# Verificar que todas las columnas existen
FEATURES = [f for f in FEATURES if f in df.columns]

# Si faltan, agregar las derivadas
for col in ['TaxiTotal', 'ElapsedDiff', 'DepTimeNorm', 'DepDelay_sq']:
    if len(FEATURES) < 20 and col in df.columns and col not in FEATURES:
        FEATURES.append(col)

X_raw = df[FEATURES].values.astype(float)
y_raw = df['ArrDelay'].values.astype(float)

m, n = X_raw.shape
print(f'Ejemplos (m): {m:,}')
print(f'Características (n): {n}')
print(f'Features: {FEATURES}')

## Parte 6 — Visualización exploratoria

Se grafican las dos relaciones más directas con `ArrDelay`:
`DepDelay` (retraso en salida) y `Distance` (distancia del vuelo).

In [ ]:
# Muestra aleatoria pequeña para graficar sin saturar
idx_plot = np.random.choice(len(y_raw), size=5000, replace=False)

fig, axes = pyplot.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(X_raw[idx_plot, FEATURES.index('DepDelay')],
                y_raw[idx_plot], alpha=0.15, s=5, color='steelblue')
axes[0].set_xlabel('Retraso en salida (DepDelay, min)')
axes[0].set_ylabel('Retraso en llegada (ArrDelay, min)')
axes[0].set_title('ArrDelay vs. DepDelay')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_raw[idx_plot, FEATURES.index('Distance')],
                y_raw[idx_plot], alpha=0.15, s=5, color='darkorange')
axes[1].set_xlabel('Distancia del vuelo (millas)')
axes[1].set_ylabel('Retraso en llegada (ArrDelay, min)')
axes[1].set_title('ArrDelay vs. Distance')
axes[1].grid(True, alpha=0.3)

pyplot.suptitle('Dataset: Airline On-Time Performance 2008', fontsize=12)
pyplot.tight_layout()
pyplot.show()

print(f'Retraso promedio en llegada: {y_raw.mean():.1f} min')
print(f'% vuelos con retraso (>0 min): {(y_raw > 0).mean()*100:.1f}%')

## Parte 7 — División Train / Validación y Normalización

Se usa 80% para entrenamiento y 20% para validación.
La normalización (media 0, std 1) se calcula **solo sobre el set de entrenamiento**
y se aplica también al set de validación para evitar *data leakage*.

In [ ]:
np.random.seed(42)
indices   = np.random.permutation(m)
split     = int(0.8 * m)
train_idx = indices[:split]
val_idx   = indices[split:]

X_train_raw = X_raw[train_idx]
X_val_raw   = X_raw[val_idx]
y_train     = y_raw[train_idx]
y_val       = y_raw[val_idx]

print(f'Entrenamiento: {len(y_train):,} ejemplos')
print(f'Validación:    {len(y_val):,} ejemplos')

In [ ]:
def normalizarCaracteristicas(X_train, X_val=None):
    """
    Normaliza con media 0 y desv. std. 1.
    Calcula mu/sigma sobre X_train y los aplica a X_val.
    """
    mu    = np.mean(X_train, axis=0)
    sigma = np.std(X_train,  axis=0)
    sigma[sigma == 0] = 1  # evita división por cero
    X_train_norm = (X_train - mu) / sigma
    X_val_norm   = (X_val - mu) / sigma if X_val is not None else None
    return X_train_norm, X_val_norm, mu, sigma

X_train_norm, X_val_norm, mu, sigma = normalizarCaracteristicas(X_train_raw, X_val_raw)
print('Normalización completada ✔')
print(f'mu    (primeras 5): {mu[:5].round(2)}')
print(f'sigma (primeras 5): {sigma[:5].round(2)}')

## Parte 8 — Funciones base del modelo

Se definen las funciones comunes a los tres modelos:
- **Hipótesis vectorizada:** h(X) = Xθ
- **Función de costo J(θ):** MSE = (1/2m) ‖Xθ − y‖²
- **Descenso de Gradiente:** θⱼ ← θⱼ − α/m · Xᵀ(Xθ−y)
- **Métricas de evaluación:** MAE, RMSE, R²

In [ ]:
def calcularCosto(X, y, theta):
    """J(θ) = (1/2m) * ||Xθ - y||². X debe incluir columna de unos."""
    m = len(y)
    e = X.dot(theta) - y
    return (1 / (2 * m)) * e.dot(e)


def descensoGradiente(X, y, theta, alpha, num_iters):
    """
    Descenso de gradiente vectorizado.
    θ ← θ - (α/m) · Xᵀ(Xθ - y)
    Retorna (theta_optimo, J_history).
    """
    m     = len(y)
    theta = theta.copy()
    J_history = []
    for _ in range(num_iters):
        e      = X.dot(theta) - y
        theta -= (alpha / m) * X.T.dot(e)
        J_history.append(calcularCosto(X, y, theta))
    return theta, J_history


def agregarColumnaUnos(X):
    """Agrega columna de 1s al inicio de X para el término de sesgo θ₀."""
    return np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)


def metricas(y_real, y_pred, nombre=''):
    """Imprime MAE, RMSE y R²."""
    mae  = np.mean(np.abs(y_pred - y_real))
    rmse = np.sqrt(np.mean((y_pred - y_real)**2))
    ss_res = np.sum((y_real - y_pred)**2)
    ss_tot = np.sum((y_real - y_real.mean())**2)
    r2   = 1 - ss_res / ss_tot
    print(f'══ {nombre} ══')
    print(f'  MAE  (error absoluto medio):  {mae:>8.2f} min')
    print(f'  RMSE (raíz error cuadrático): {rmse:>8.2f} min')
    print(f'  R²   (coef. determinación):   {r2:>8.4f}')
    return mae, rmse, r2

print('Funciones base definidas ✔')

---
# MODELO 1 — Regresión Lineal Multivariable
## Descenso de Gradiente con 20 características

Se entrena la hipótesis lineal estándar `h(x) = θ₀ + θ₁x₁ + … + θₙxₙ`
con todas las características normalizadas.
Primero se busca el **alpha óptimo** comparando la convergencia de varias tasas.

In [ ]:
X_tr1 = agregarColumnaUnos(X_train_norm)
X_vl1 = agregarColumnaUnos(X_val_norm)
n_cols = X_tr1.shape[1]

alphas_prueba = [0.001, 0.01, 0.05, 0.1, 0.3]
iters_prueba  = 300

pyplot.figure(figsize=(10, 5))
print(f'{"Alpha":<10} {"Costo final":>15}')
print('-' * 28)
for a in alphas_prueba:
    th0  = np.zeros(n_cols)
    _, Jh = descensoGradiente(X_tr1, y_train, th0, a, iters_prueba)
    print(f'{a:<10} {Jh[-1]:>15.4f}')
    pyplot.plot(Jh, label=f'α={a}')

pyplot.xlabel('Iteraciones')
pyplot.ylabel('Costo J')
pyplot.title('Modelo 1 — Comparación de tasas de aprendizaje')
pyplot.yscale('log')
pyplot.legend()
pyplot.grid(True, alpha=0.3)
pyplot.tight_layout()
pyplot.show()

In [ ]:
# Entrenamiento final Modelo 1 con alpha óptimo
alpha1     = 0.1
num_iters1 = 1500
theta1, J_history1 = descensoGradiente(X_tr1, y_train, np.zeros(n_cols), alpha1, num_iters1)

print(f'Entrenamiento Modelo 1 completado ✔')
print(f'  alpha = {alpha1}  |  iteraciones = {num_iters1}')
print(f'  Costo inicial: {J_history1[0]:,.2f}')
print(f'  Costo final:   {J_history1[-1]:,.2f}')

fig, axes = pyplot.subplots(1, 2, figsize=(13, 4))
axes[0].plot(J_history1, color='steelblue', lw=2)
axes[0].set_title('Modelo 1 — Convergencia (escala lineal)')
axes[0].set_xlabel('Iteraciones')
axes[0].set_ylabel('Costo J')
axes[0].grid(True, alpha=0.3)

axes[1].plot(J_history1, color='steelblue', lw=2)
axes[1].set_title('Modelo 1 — Convergencia (escala logarítmica)')
axes[1].set_xlabel('Iteraciones')
axes[1].set_ylabel('Costo J (log)')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

pyplot.tight_layout()
pyplot.show()

### Validación Modelo 1

In [ ]:
y_pred1_train = X_tr1.dot(theta1)
y_pred1_val   = X_vl1.dot(theta1)

print('--- Entrenamiento ---')
mae1_tr, rmse1_tr, r2_1_tr = metricas(y_train, y_pred1_train, 'M1 Train')
print()
print('--- Validación ---')
mae1_vl, rmse1_vl, r2_1_vl = metricas(y_val, y_pred1_val, 'M1 Validación')

idx_v = np.random.choice(len(y_val), 3000, replace=False)
pyplot.figure(figsize=(7, 5))
pyplot.scatter(y_val[idx_v], y_pred1_val[idx_v], alpha=0.2, s=6, color='steelblue')
lim = np.percentile(np.concatenate([y_val, y_pred1_val]), 99)
pyplot.plot([-60, lim], [-60, lim], 'r--', lw=1.5, label='Predicción perfecta')
pyplot.xlabel('ArrDelay real (min)')
pyplot.ylabel('ArrDelay predicho (min)')
pyplot.title(f'Modelo 1 — Predicho vs. Real | R²={r2_1_vl:.4f}')
pyplot.legend()
pyplot.tight_layout()
pyplot.show()

---
# MODELO 2 — Regresión Polinómica

Se agregan **términos cuadráticos** de las características con mayor correlación con `ArrDelay`
(`DepDelay`, `CarrierDelay`, `WeatherDelay`, `LateAircraftDelay`).
Esto permite capturar relaciones no lineales entre las variables y el retraso de llegada.

In [ ]:
def crearFeaturesPol(X_norm, indices_cuad):
    """
    Agrega columnas x² para los índices indicados.
    Recibe X ya normalizado.
    """
    cols = [X_norm]
    for idx in indices_cuad:
        cols.append((X_norm[:, idx] ** 2).reshape(-1, 1))
    return np.hstack(cols)

# Índices de: DepDelay(6), CarrierDelay(13), WeatherDelay(14), LateAircraftDelay(17)
# (ajustar según el orden real de FEATURES si cambió)
IDX_CUAD = []
for nombre in ['DepDelay', 'CarrierDelay', 'WeatherDelay', 'LateAircraftDelay']:
    if nombre in FEATURES:
        IDX_CUAD.append(FEATURES.index(nombre))

X_train_pol = crearFeaturesPol(X_train_norm, IDX_CUAD)
X_val_pol   = crearFeaturesPol(X_val_norm,   IDX_CUAD)

X_tr2 = agregarColumnaUnos(X_train_pol)
X_vl2 = agregarColumnaUnos(X_val_pol)

print(f'Dimensión X_train polinómica: {X_tr2.shape}')
print(f'  ({n} features originales + {len(IDX_CUAD)} cuadráticas + 1 sesgo = {X_tr2.shape[1]} columnas)')

In [ ]:
alpha2     = 0.1
num_iters2 = 1500
theta2, J_history2 = descensoGradiente(X_tr2, y_train, np.zeros(X_tr2.shape[1]), alpha2, num_iters2)

print(f'Entrenamiento Modelo 2 completado ✔')
print(f'  Costo inicial: {J_history2[0]:,.2f}')
print(f'  Costo final:   {J_history2[-1]:,.2f}')

fig, axes = pyplot.subplots(1, 2, figsize=(13, 4))
axes[0].plot(J_history2, color='darkorange', lw=2)
axes[0].set_title('Modelo 2 — Convergencia (escala lineal)')
axes[0].set_xlabel('Iteraciones')
axes[0].set_ylabel('Costo J')
axes[0].grid(True, alpha=0.3)

axes[1].plot(J_history2, color='darkorange', lw=2)
axes[1].set_title('Modelo 2 — Convergencia (escala logarítmica)')
axes[1].set_xlabel('Iteraciones')
axes[1].set_ylabel('Costo J (log)')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

pyplot.tight_layout()
pyplot.show()

### Validación Modelo 2

In [ ]:
y_pred2_train = X_tr2.dot(theta2)
y_pred2_val   = X_vl2.dot(theta2)

print('--- Entrenamiento ---')
mae2_tr, rmse2_tr, r2_2_tr = metricas(y_train, y_pred2_train, 'M2 Train')
print()
print('--- Validación ---')
mae2_vl, rmse2_vl, r2_2_vl = metricas(y_val, y_pred2_val, 'M2 Validación')

pyplot.figure(figsize=(7, 5))
pyplot.scatter(y_val[idx_v], y_pred2_val[idx_v], alpha=0.2, s=6, color='darkorange')
pyplot.plot([-60, lim], [-60, lim], 'r--', lw=1.5, label='Predicción perfecta')
pyplot.xlabel('ArrDelay real (min)')
pyplot.ylabel('ArrDelay predicho (min)')
pyplot.title(f'Modelo 2 — Predicho vs. Real | R²={r2_2_vl:.4f}')
pyplot.legend()
pyplot.tight_layout()
pyplot.show()

---
# MODELO 3 — Ecuación Normal

La ecuación normal calcula θ de forma **analítica exacta** en un solo paso:

$$\theta = (X^T X)^{-1} X^T \vec{y}$$

No requiere normalización ni elegir alpha ni iterar.
Se usa `np.linalg.pinv` (pseudoinversa) en lugar de `inv` para mayor estabilidad numérica
con datasets grandes.

**Nota:** con 2M de filas la ecuación normal sería muy costosa en memoria.
Se aplica sobre los mismos 200,000 ejemplos de entrenamiento sin normalizar.

In [ ]:
def ecuacionNormal(X, y):
    """θ = (XᵀX)⁻¹ Xᵀy  — usa pseudoinversa para estabilidad numérica."""
    return np.linalg.pinv(X.T.dot(X)).dot(X.T).dot(y)

X_tr3 = agregarColumnaUnos(X_train_raw)   # sin normalizar
X_vl3 = agregarColumnaUnos(X_val_raw)

print('Calculando ecuación normal...')
theta3 = ecuacionNormal(X_tr3, y_train)

J_en_train = calcularCosto(X_tr3, y_train, theta3)
J_en_val   = calcularCosto(X_vl3, y_val,   theta3)

print(f'Ecuación Normal calculada ✔')
print(f'  Costo J (train):      {J_en_train:,.4f}')
print(f'  Costo J (validación): {J_en_val:,.4f}')

# Comparación visual del costo (train vs val)
pyplot.figure(figsize=(5, 4))
pyplot.bar(['Train', 'Validación'], [J_en_train, J_en_val],
           color=['steelblue', 'darkorange'], edgecolor='k', width=0.5)
pyplot.ylabel('Costo J')
pyplot.title('Modelo 3 (Ec. Normal) — Costo J\nTrain vs. Validación')
pyplot.grid(True, axis='y', alpha=0.3)
pyplot.tight_layout()
pyplot.show()

### Validación Modelo 3

In [ ]:
y_pred3_train = X_tr3.dot(theta3)
y_pred3_val   = X_vl3.dot(theta3)

print('--- Entrenamiento ---')
mae3_tr, rmse3_tr, r2_3_tr = metricas(y_train, y_pred3_train, 'M3 Train')
print()
print('--- Validación ---')
mae3_vl, rmse3_vl, r2_3_vl = metricas(y_val, y_pred3_val, 'M3 Validación')

pyplot.figure(figsize=(7, 5))
pyplot.scatter(y_val[idx_v], y_pred3_val[idx_v], alpha=0.2, s=6, color='seagreen')
pyplot.plot([-60, lim], [-60, lim], 'r--', lw=1.5, label='Predicción perfecta')
pyplot.xlabel('ArrDelay real (min)')
pyplot.ylabel('ArrDelay predicho (min)')
pyplot.title(f'Modelo 3 — Predicho vs. Real | R²={r2_3_vl:.4f}')
pyplot.legend()
pyplot.tight_layout()
pyplot.show()

---
# Comparación de los tres modelos

Se resumen las métricas de **validación** de los tres modelos para identificar
cuál predice mejor el retraso de llegada.

In [ ]:
nombres   = ['M1\nLineal\nMultivariable', 'M2\nPolinómica', 'M3\nEc. Normal']
rmse_vals = [rmse1_vl, rmse2_vl, rmse3_vl]
r2_vals   = [r2_1_vl,  r2_2_vl,  r2_3_vl]
mae_vals  = [mae1_vl,  mae2_vl,  mae3_vl]

print(f'{"Modelo":<28} {"MAE (min)":>12} {"RMSE (min)":>12} {"R²":>10}')
print('-' * 66)
for nm, mae_v, rmse_v, r2_v in zip(nombres, mae_vals, rmse_vals, r2_vals):
    nm_clean = nm.replace('\n', ' ')
    print(f'{nm_clean:<28} {mae_v:>12.2f} {rmse_v:>12.2f} {r2_v:>10.4f}')

colores = ['steelblue', 'darkorange', 'seagreen']
fig, axes = pyplot.subplots(1, 3, figsize=(14, 4))

axes[0].bar(nombres, mae_vals, color=colores)
axes[0].set_title('MAE (min) — menor es mejor')
axes[0].set_ylabel('minutos')

axes[1].bar(nombres, rmse_vals, color=colores)
axes[1].set_title('RMSE (min) — menor es mejor')
axes[1].set_ylabel('minutos')

axes[2].bar(nombres, r2_vals, color=colores)
axes[2].set_title('R² — mayor es mejor')
axes[2].set_ylim(0, 1)

pyplot.suptitle('Comparación de modelos — Set de Validación', fontsize=12)
pyplot.tight_layout()
pyplot.show()

---
# Predicciones — 100 vuelos

Se generan **100 predicciones** usando el modelo con mayor R² en validación.
Se muestra el retraso predicho vs. el real para cada vuelo.

In [ ]:
mejor_idx    = int(np.argmax(r2_vals))
nombre_mejor = nombres[mejor_idx].replace('\n', ' ')
print(f'Mejor modelo: {nombre_mejor}  (R²={r2_vals[mejor_idx]:.4f})')

muestra_100 = np.random.choice(len(y_val), size=100, replace=False)

if mejor_idx == 0:
    y_100_pred = X_vl1[muestra_100].dot(theta1)
elif mejor_idx == 1:
    y_100_pred = X_vl2[muestra_100].dot(theta2)
else:
    y_100_pred = X_vl3[muestra_100].dot(theta3)

y_100_real = y_val[muestra_100]

print(f'\n{"#":<5} {"ArrDelay Real (min)":>22} {"ArrDelay Predicho (min)":>25} {"Error abs":>12}')
print('-' * 68)
for i, (real, pred) in enumerate(zip(y_100_real[:20], y_100_pred[:20])):
    print(f'{i+1:<5} {real:>22.1f} {pred:>25.1f} {abs(pred-real):>12.1f}')
print('  ... (mostrando primeras 20 de 100)')

In [ ]:
errores_100 = np.abs(y_100_pred - y_100_real)

fig, axes = pyplot.subplots(1, 2, figsize=(14, 5))

# Predicho vs Real
axes[0].scatter(y_100_real, y_100_pred, s=60, alpha=0.7,
                color='royalblue', edgecolors='k', linewidths=0.4)
lim_pred = np.percentile(np.concatenate([y_100_real, y_100_pred]), 99)
axes[0].plot([-60, lim_pred], [-60, lim_pred], 'r--', lw=1.5, label='Predicción perfecta')
axes[0].set_xlabel('ArrDelay real (min)')
axes[0].set_ylabel('ArrDelay predicho (min)')
axes[0].set_title(f'100 Predicciones — {nombre_mejor}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error absoluto por predicción
axes[1].bar(range(100), errores_100, color='steelblue', alpha=0.7)
axes[1].axhline(errores_100.mean(), color='red', lw=2, linestyle='--',
                label=f'Error medio: {errores_100.mean():.1f} min')
axes[1].set_xlabel('Predicción #')
axes[1].set_ylabel('Error absoluto (min)')
axes[1].set_title('Error absoluto por predicción')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

pyplot.tight_layout()
pyplot.show()

print(f'\nResumen de las 100 predicciones:')
print(f'  Error absoluto medio:         {errores_100.mean():.1f} min')
print(f'  Error absoluto mediana:       {np.median(errores_100):.1f} min')
print(f'  Predicciones con error <5min: {(errores_100 < 5).sum()} / 100')
print(f'  Predicciones con error <10min:{(errores_100 < 10).sum()} / 100')

---
# Conclusiones

Se entrenaron y compararon tres modelos de regresión lineal sobre el dataset
**Airline On-Time Performance 2008** (Data Expo 2009, DOT/BTS),
con ~200,000 vuelos de entrenamiento y 20 características numéricas.

| Modelo | Método | Features |
|--------|--------|----------|
| M1 | Descenso de Gradiente | 20 lineales |
| M2 | Descenso de Gradiente | 20 + 4 cuadráticas |
| M3 | Ecuación Normal | 20 sin normalizar |

**Observaciones:**
- La variable `DepDelay` (retraso en salida) tiene una **correlación muy alta** con `ArrDelay`.
  Los modelos que la incluyen tienden a alcanzar R² > 0.90.
- La **ecuación normal** (M3) produce la solución exacta y suele igualar o superar
  al descenso de gradiente con suficientes iteraciones.
- La **regresión polinómica** (M2) captura relaciones no lineales de los retrasos
  por clima y por carrier, mejorando ligeramente sobre M1.
- El **descenso de gradiente** requiere normalización y ajuste de alpha,
  pero es escalable a conjuntos de datos mucho más grandes que la ecuación normal.